In [ ]:
# FIRST CELL — imports and settings
import os
import sys
sys.path.append('../../src')  # repo-relative: notebooks/pipeline/../../src

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

from config import *

# Load data here to satisfy the shape print requirement in the first cell
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print(f"Initial train shape: {train_df.shape}")
print(f"Initial test shape: {test_df.shape}")

In [2]:
# CELL 1 — Load raw data
# Data was loaded in the first cell, re-confirming shapes and displaying rows as requested.

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}\n")

print("Train first 3 rows:")
display(train_df.head(3))

print("\nTrain columns:")
print(train_df.columns.tolist())
print("\nTest columns:")
print(test_df.columns.tolist())

diff = set(train_df.columns) - set(test_df.columns)
print(f"\nColumns in train but not in test: {diff}")

Train shape: (77299, 11)
Test shape: (41778, 10)

Train first 3 rows:


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny



Train columns:
['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Test columns:
['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Columns in train but not in test: {'demand'}


In [3]:
# CELL 2 — Drop Index column
# The Index column is a row identifier with no predictive value.
train_df = train_df.drop(columns=[ID_COL])
test_df = test_df.drop(columns=[ID_COL])

print(f"Train shape after dropping {ID_COL}: {train_df.shape}")
print(f"Test shape after dropping {ID_COL}: {test_df.shape}")

Train shape after dropping Index: (77299, 10)
Test shape after dropping Index: (41778, 9)


In [ ]:
# CELL 3 — Parse timestamp
# hour and minute are kept as plain integers here rather than jumping straight
# to sin/cos — that cyclical encoding only happens in step05, once hour has
# been used on its own for a few EDA-adjacent sanity checks below. minute
# turns out to carry no signal (every geohash is sampled at all 4 quarter-hour
# marks) and gets dropped along with everything not in FINAL_FEATURES at step08.
def extract_hour(ts):
    return int(str(ts).split(':')[0])

def extract_minute(ts):
    return int(str(ts).split(':')[1])

train_df['hour'] = train_df[TIMESTAMP_COL].apply(extract_hour)
train_df['minute'] = train_df[TIMESTAMP_COL].apply(extract_minute)

test_df['hour'] = test_df[TIMESTAMP_COL].apply(extract_hour)
test_df['minute'] = test_df[TIMESTAMP_COL].apply(extract_minute)

train_df = train_df.drop(columns=[TIMESTAMP_COL])
test_df = test_df.drop(columns=[TIMESTAMP_COL])

print("First 5 rows of hour and minute columns (Train):")
print(train_df[['hour', 'minute']].head())

print("\nUnique hours in Train:")
print(sorted(train_df['hour'].unique()))
print("\nUnique minutes in Train:")
print(sorted(train_df['minute'].unique()))

In [5]:
# CELL 4 — Impute RoadType and Weather using sklearn SimpleImputer
print("Null counts BEFORE imputation (Train):")
print(train_df[['RoadType', 'Weather']].isnull().sum())
print("\nNull counts BEFORE imputation (Test):")
print(test_df[['RoadType', 'Weather']].isnull().sum())

imputer = SimpleImputer(strategy='most_frequent')

# Fit ONLY on train to prevent data leakage
imputer.fit(train_df[['RoadType', 'Weather']])

train_df[['RoadType', 'Weather']] = imputer.transform(train_df[['RoadType', 'Weather']])
test_df[['RoadType', 'Weather']]  = imputer.transform(test_df[['RoadType', 'Weather']])

print("\nNull counts AFTER imputation (Train):")
print(train_df[['RoadType', 'Weather']].isnull().sum())
print("\nNull counts AFTER imputation (Test):")
print(test_df[['RoadType', 'Weather']].isnull().sum())

Null counts BEFORE imputation (Train):
RoadType    600
Weather     797
dtype: int64

Null counts BEFORE imputation (Test):
RoadType    324
Weather     431
dtype: int64

Null counts AFTER imputation (Train):
RoadType    0
Weather     0
dtype: int64

Null counts AFTER imputation (Test):
RoadType    0
Weather     0
dtype: int64


In [6]:
# CELL 5 — Verify no nulls remain in non-temperature columns
print("Train Null Summary:")
train_nulls = train_df.isnull().sum()
print(train_nulls[train_nulls > 0])

print("\nTest Null Summary:")
test_nulls = test_df.isnull().sum()
print(test_nulls[test_nulls > 0])

unexpected_train = train_nulls.drop(['Temperature', 'demand'], errors='ignore')
if unexpected_train.sum() > 0:
    print("\nWARNING: Unexpected nulls found in train non-temperature columns!")
    
unexpected_test = test_nulls.drop('Temperature', errors='ignore')
if unexpected_test.sum() > 0:
    print("WARNING: Unexpected nulls found in test non-temperature columns!")

Train Null Summary:
Temperature    2495
dtype: int64

Test Null Summary:


Temperature    1349
dtype: int64

In [7]:
# CELL 6 — Basic feature confirmation
print("RoadType value counts (Train):")
print(train_df['RoadType'].value_counts())

print("\nWeather value counts (Train):")
print(train_df['Weather'].value_counts())

print("\nLargeVehicles value counts (Train):")
print(train_df['LargeVehicles'].value_counts())

print("\nLandmarks value counts (Train):")
print(train_df['Landmarks'].value_counts())

print("\nHour column stats (Train):")
print(f"Min: {train_df['hour'].min()}, Max: {train_df['hour'].max()}, Mean: {train_df['hour'].mean():.2f}")

RoadType value counts (Train):
RoadType
Residential    69830
Street          3909
Highway         3560
Name: count, dtype: int64

Weather value counts (Train):
Weather
Sunny    28514
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64

LargeVehicles value counts (Train):
LargeVehicles
Not Allowed    50673
Allowed        26626
Name: count, dtype: int64

Landmarks value counts (Train):
Landmarks
Yes    52042
No     25257
Name: count, dtype: int64

Hour column stats (Train):
Min: 0, Max: 23, Mean: 9.10


In [8]:
# CELL 7 — Save cleaned data
train_save_path = os.path.join(PROC_DIR, 'train_step01.csv')
test_save_path = os.path.join(PROC_DIR, 'test_step01.csv')

train_df.to_csv(train_save_path, index=False)
test_df.to_csv(test_save_path, index=False)

print(f"Saved cleaned train to: {train_save_path}")
print(f"Train shape: {train_df.shape}")

print(f"\nSaved cleaned test to: {test_save_path}")
print(f"Test shape: {test_df.shape}")

Saved cleaned train to: C:\traffic-demand-final\pipelining\processed\train_step01.csv
Train shape: (77299, 11)

Saved cleaned test to: C:\traffic-demand-final\pipelining\processed\test_step01.csv
Test shape: (41778, 10)
